# 5.21 Hamiltonian Monte Carlo & NUTS

**Lesson tagline.** Gradients let MCMC move far while preserving high acceptance.

HMC improves continuous MCMC by following posterior geometry with leapfrog integration. NUTS-style turning checks prevent wasteful trajectories when geometry bends back on itself. Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 521
rng = np.random.default_rng(SEED)

def effective_sample_size(series):
    x = np.asarray(series, dtype=float)
    x = x - x.mean()
    denom = np.dot(x, x)
    if denom == 0:
        return float(len(x))
    values = []
    for lag in range(1, min(200, len(x) - 1)):
        corr = np.dot(x[:-lag], x[lag:]) / denom
        if corr <= 0:
            break
        values.append(corr)
    tau = 1.0 + 2.0 * np.sum(values)
    return len(x) / tau

def gaussian_U_grad(mean, cov):
    inv = np.linalg.inv(cov)
    def U(q):
        diff = q - mean
        return 0.5 * float(diff @ inv @ diff)
    def grad_U(q):
        return inv @ (q - mean)
    return U, grad_U

def mixture_U_grad_1d(means, scales, weights):
    def components(q):
        q0 = float(np.ravel(q)[0])
        vals = []
        grads = []
        for mean, scale, weight in zip(means, scales, weights):
            z = (q0 - mean) / scale
            density = weight * np.exp(-0.5 * z ** 2) / (np.sqrt(2 * np.pi) * scale)
            vals.append(density)
            grads.append(density * (-(q0 - mean) / (scale ** 2)))
        return np.array(vals), np.array(grads)
    def U(q):
        vals, _ = components(q)
        return -np.log(np.sum(vals) + 1e-300)
    def grad_U(q):
        vals, grads = components(q)
        grad_log = np.sum(grads) / (np.sum(vals) + 1e-300)
        return np.array([-grad_log])
    return U, grad_U

def leapfrog(q, p, grad_U, epsilon, L, inv_mass=None):
    q = np.asarray(q, dtype=float).copy()
    p = np.asarray(p, dtype=float).copy()
    if inv_mass is None:
        inv_mass = np.ones_like(q)
    p = p - 0.5 * epsilon * grad_U(q)
    for step in range(L):
        q = q + epsilon * inv_mass * p
        if step != L - 1:
            p = p - epsilon * grad_U(q)
    p = p - 0.5 * epsilon * grad_U(q)
    return q, -p

def hamiltonian(q, p, U, inv_mass=None):
    if inv_mass is None:
        inv_mass = np.ones_like(p)
    kinetic = 0.5 * np.sum(inv_mass * p ** 2)
    return U(q) + kinetic

def run_hmc(U, grad_U, dim, n_steps, epsilon, L, random_state):
    q = np.zeros(dim)
    samples = []
    accepted = 0
    energy_errors = []
    for _ in range(n_steps):
        p = random_state.normal(size=dim)
        current_H = hamiltonian(q, p, U)
        q_new, p_new = leapfrog(q, p, grad_U, epsilon, L)
        proposed_H = hamiltonian(q_new, p_new, U)
        delta = proposed_H - current_H
        if np.log(random_state.random()) < -delta:
            q = q_new
            accepted += 1
        energy_errors.append(delta)
        samples.append(q.copy())
    return np.array(samples), accepted / n_steps, np.array(energy_errors)

def build_hmc_ladder():
    ladder = []
    U1, G1 = gaussian_U_grad(np.zeros(1), np.eye(1))
    U2, G2 = gaussian_U_grad(np.zeros(4), np.diag([0.5, 1.0, 1.5, 2.0]))
    U3, G3 = mixture_U_grad_1d([-2.0, 2.0], [0.45, 0.55], [0.5, 0.5])
    U4, G4 = gaussian_U_grad(np.zeros(2), np.array([[1.0, 0.93], [0.93, 1.1]]))
    U5, G5 = gaussian_U_grad(np.zeros(8), np.diag([0.02, 0.04, 0.08, 0.2, 0.7, 1.4, 3.0, 7.0]))
    ladder.append({"name": "D1 1-D Gaussian", "dim": 1, "U": U1, "grad": G1, "truth": 0.0, "epsilon": 0.2, "L": 5})
    ladder.append({"name": "D2 4-D diagonal", "dim": 4, "U": U2, "grad": G2, "truth": 0.0, "epsilon": 0.18, "L": 7})
    ladder.append({"name": "D3 bimodal", "dim": 1, "U": U3, "grad": G3, "truth": 0.0, "epsilon": 0.08, "L": 18})
    ladder.append({"name": "D4 correlated 2-D", "dim": 2, "U": U4, "grad": G4, "truth": 0.0, "epsilon": 0.08, "L": 20})
    ladder.append({"name": "D5 ill-conditioned 8-D", "dim": 8, "U": U5, "grad": G5, "truth": 0.0, "epsilon": 0.025, "L": 35})
    return ladder

## The concept, built once (D1)
HMC augments position $q$ with momentum $p$ and approximately conserves
$$H(q,p)=U(q)+K(p),\quad p\leftarrow p-\frac{\epsilon}{2}\nabla U(q),\quad q\leftarrow q+\epsilon M^{-1}p.$$
The first leapfrog half-step is small enough that the Metropolis correction accepts with high probability.

In [ ]:
def lesson_grad_U(q):
    return np.asarray(q)

q0 = np.array([1.0])
p0 = np.array([0.3])
epsilon = 0.2
p_half = p0 - 0.5 * epsilon * lesson_grad_U(q0)
q1 = q0 + epsilon * p_half
assert abs(p_half[0] - 0.2) < 1e-12
assert abs(q1[0] - 1.04) < 1e-12
print("p_half", p_half[0])
print("q1", q1[0])

Now use the reusable leapfrog integrator and check the lesson's energy claim. For a unit Gaussian target, one short trajectory changes energy by less than $0.01$, so acceptance is above $0.99$. NUTS adds the stop rule: stop when displacement dot momentum becomes negative.

In [ ]:
U_lesson, G_lesson = gaussian_U_grad(np.zeros(1), np.eye(1))
q_new, p_new = leapfrog(q0, p0, G_lesson, 0.2, 1)
delta_H = hamiltonian(q_new, p_new, U_lesson) - hamiltonian(q0, p0, U_lesson)
accept_prob = min(1.0, float(np.exp(-delta_H)))
turning = float((q_new - q0) @ (-p_new)) < 0.0
assert abs(delta_H) < 0.01
assert accept_prob > 0.99
print("energy change", delta_H)
print("acceptance", accept_prob)
print("NUTS stop would trigger", turning)

## The dataset ladder
The F10 ladder is continuous here: 1-D Gaussian, 4-D diagonal Gaussian, bimodal mixture, correlated 2-D geometry, and a higher-dimensional ill-conditioned posterior.

In [ ]:
ladder = build_hmc_ladder()
for i, rung in enumerate(ladder, start=1):
    print(i, rung["name"], "dim=", rung["dim"], "epsilon=", rung["epsilon"])

## Run the SAME method across D1-D5
Collect the plan metric on every rung from D1–D5 with the same algorithmic implementation, then print a compact per-rung table.

In [ ]:
checkpoints = [50, 100, 200, 400, 800]
hmc_results = {}
print("rung | marginal error | ESS per grad | acceptance")
for rung in ladder:
    samples, accept_rate, energy_errors = run_hmc(rung["U"], rung["grad"], rung["dim"], checkpoints[-1], rung["epsilon"], rung["L"], rng)
    errors = []
    for n in checkpoints:
        errors.append(abs(samples[:n, 0].mean() - rung["truth"]))
    ess = effective_sample_size(samples[:, 0])
    ess_per_grad = ess / (checkpoints[-1] * rung["L"])
    hmc_results[rung["name"]] = {"samples": samples, "errors": errors, "ess": ess, "ess_per_grad": ess_per_grad, "accept": accept_rate, "energy": energy_errors}
    print(f"{rung['name']} | {errors[-1]:.4f} | {ess_per_grad:.4f} | {accept_rate:.2f}")

## Results visualization
The closing figure has target/sample panels on top and the requested error-vs-iteration or error-vs-sample curve below.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for ax, rung in zip(axes[0], ladder):
    samples = hmc_results[rung["name"]]["samples"]
    if rung["dim"] == 1:
        ax.hist(samples[:, 0], bins=30, density=True, alpha=0.75)
    else:
        ax.scatter(samples[:, 0], samples[:, 1], s=4, alpha=0.4)
    ax.axvline(rung["truth"], color="black", linestyle="--")
    ax.set_title(rung["name"], fontsize=8)
for ax, rung in zip(axes[1], ladder):
    ax.plot(checkpoints, hmc_results[rung["name"]]["errors"], marker="o")
    ax.set_xscale("log")
    ax.set_title("marginal error", fontsize=8)
fig.tight_layout()
plt.show()

## Pitfall on the hardest rung
A too-large step size on the ill-conditioned D5 target makes leapfrog energy error explode and acceptance collapse. The fix is a smaller step size or NUTS-style/adaptive trajectory control that respects the narrowest direction.

In [ ]:
d5 = ladder[-1]
bad_samples, bad_accept, bad_energy = run_hmc(d5["U"], d5["grad"], d5["dim"], 400, 0.16, d5["L"], rng)
good_samples, good_accept, good_energy = run_hmc(d5["U"], d5["grad"], d5["dim"], 400, d5["epsilon"], d5["L"], rng)
print("too-large step acceptance", round(bad_accept, 3), "median |dH|", round(float(np.median(np.abs(bad_energy))), 3))
print("fixed step acceptance", round(good_accept, 3), "median |dH|", round(float(np.median(np.abs(good_energy))), 3))
displacement = good_samples[-1] - good_samples[0]
momentum = rng.normal(size=d5["dim"])
print("NUTS turning statistic", float(displacement @ momentum))

## Evaluate it + Practice
- **Metric.** ESS per gradient evaluation and marginal mean error.
- **No-skill baseline.** Random-walk Metropolis with the same number of target evaluations.
- **Cheap sanity check.** Energy error should be centered near zero for a stable step size.
- **Ablation.** Set epsilon too large and record divergences or rejected proposals.
- **Failure signals.** Acceptance collapses, median absolute energy error grows, or chains stick in one mode.

### Practice prompts
1. Try a diagonal mass matrix for D5.
2. Compare short and long trajectories on D4.
3. Implement the NUTS U-turn check over a saved trajectory.

In [ ]:
# Your code here

In [ ]:
# Your code here

In [ ]:
# Your code here